# Clasificador Bayesiano

#### Sandra Marlene Cobian Aquino
#### Guillermo Christopher Mendieta Arreola

In [ ]:
import pandas as pd
import numpy as np

## 1. Procesamiento del Dataset

In [ ]:
# preprocessing
class Preprocessor:

  def __init__(self, target="Clase"):
    self.target = target
    self.df = None
    self.train = None
    self.test = None

  def load_data(self, file_path):
    # load the csv
    self.df=pd.read_csv(file_path)

  def age_to_bin(self):
    # binning age column
    age_bins = [17, 29, 59, 75];
    age_labels = [0, 1, 2];
    self.df['age-bin'] = pd.cut(self.df['Edad'], bins=age_bins, labels=age_labels, include_lowest=True)

  def income_to_bin(self):
    # binning income column
    income_bins = [9999, 19999, 29999, 39999, 49999, 60000]
    income_labels = [0, 1, 2, 3, 4]
    self.df['income-bin'] = pd.cut(self.df['Ingreso'], bins=income_bins, labels=income_labels, include_lowest=True)

  def train_test_split(self, ratio=0.8):
    # separate train and test df
    total_rows = self.df.shape[0]
    train_size = int(total_rows*ratio)
    self.train = self.df[0:train_size]
    self.test = self.df[train_size:]

  def run(self, file_path, ratio=0.8):
    # run the preprocessing routine
    self.load_data(file_path)
    self.age_to_bin()
    self.income_to_bin()
    self.train_test_split(ratio)
    return self.train, self.test

In [ ]:
prep = Preprocessor()

prep.load_data("dataset_walk_drive_publicado.csv")
print(prep.df.head())
print(prep.df.shape)
print(prep.df.columns)

prep.age_to_bin()
print(prep.df[['Edad', 'age-bin']].head(10))

prep.income_to_bin()
print(prep.df[['Ingreso', 'income-bin']].head(10))

print(prep.df.head())

prep.train_test_split()
print("Train:", prep.train.shape)
print("Test:", prep.test.shape)

   Edad  Ingreso    Clase
0    57    32961  Conduce
1    29    32610   Camina
2    25    30051   Camina
3    54    27654  Conduce
4    58    15846  Conduce
(100, 3)
Index(['Edad', 'Ingreso', 'Clase'], dtype='object')
   Edad age-bin
0    57       1
1    29       0
2    25       0
3    54       1
4    58       1
5    25       0
6    41       1
7    23       0
8    69       2
9    36       1
   Ingreso income-bin
0    32961          2
1    32610          2
2    30051          2
3    27654          1
4    15846          0
5    25793          1
6    26572          1
7    21977          1
8    28387          1
9    34040          2
   Edad  Ingreso    Clase age-bin income-bin
0    57    32961  Conduce       1          2
1    29    32610   Camina       0          2
2    25    30051   Camina       0          2
3    54    27654  Conduce       1          1
4    58    15846  Conduce       1          0
Train: (80, 5)
Test: (20, 5)


In [ ]:
prep = Preprocessor(target="Clase")
train, test = prep.run("dataset_walk_drive_publicado.csv")

## 2. Calculo de Probabilidades

In [ ]:
# bayes classifier
class BayesClassifier:

  def __init__(self):
    self.class_priors = {}
    self.joint_probabilities = {}
    self.class_counts = {}
    self.age_bins = [0, 1, 2]
    self.income_bins = [0, 1, 2, 3, 4]
    self.num_combinations = len(self.age_bins) * len(self.income_bins)
    self.alpha = 1

  def fit(self, train_data):
    # calculate P(C)
    self.calculate_priors(train_data)

    # calculate P(x|C)
    self.calculate_joint_probabilities(train_data)

  def calculate_priors(self, train_data):
    # count the registers of each class
    counts = train_data["Clase"].value_counts()

    # save the registers of the classes
    self.class_counts = counts.to_dict()

    # get the total registers in train df
    total = len(train_data)

    # calculate the priori of each class
    for class_name in self.class_counts:
      self.class_priors[class_name] = (self.class_counts[class_name] / total)

  def calculate_joint_probabilities(self, train_data):
    # traverse each class in train df
    for class_name in self.class_counts:

      # filter registers that belong to that class
      class_data = train_data[train_data["Clase"] == class_name]

      # traverse each age and income bin
      for age_bin in self.age_bins:
        for income_bin in self.income_bins:
          # count registers that coincide with age bin and income bin
          count = len(class_data[(class_data["age-bin"] == age_bin)& (class_data["income-bin"] == income_bin)])

          # calculate confitional probability with laplace smoothing
          probability = (count + self.alpha) / (self.class_counts[class_name] + self.alpha * self.num_combinations)

          # save the result with a tuple as a key
          self.joint_probabilities[(age_bin, income_bin, class_name)] = probability

  def evidence(self, age_bin, income_bin):
    evidence_probability = 0

    # sum each class' contribution
    for class_name in self.class_priors:

      # calculate P(x|C)
      conditional_probability = self.joint_probabilities[(age_bin, income_bin, class_name)]

      # calculate P(C)
      prior_probability = self.class_priors[class_name]

      # multiply P(x|C) * P(C)
      evidence_probability += (conditional_probability * prior_probability)

    return evidence_probability

  def posterior(self, age_bin, income_bin, class_name):
    # calculate P(x|C)
    conditional_probability = self.joint_probabilities[(age_bin, income_bin, class_name)]

    # calculate P(C)
    prior_probability = self.class_priors[class_name]

    # calculate P(x)
    evidence_probability = self.evidence(age_bin, income_bin)

    # apply Bayes
    return (conditional_probability * prior_probability) / evidence_probability

  def predict(self, age_bin, income_bin):
    # calculate posterior probabilities
    prob_caminar = self.posterior(age_bin,income_bin,"Camina")
    prob_conduce = self.posterior(age_bin,income_bin,"Conduce")

    # select the clase with superiori probability
    if prob_caminar > prob_conduce:
      return "Camina"
    else:
      return "Conduce"

  def predict_proba(self, age_bin, income_bin):
    probabilities = {}

    # calculate the posterior probability of each class
    for class_name in self.class_priors:
      probabilities[class_name] = self.posterior(age_bin, income_bin, class_name)

    return probabilities

In [ ]:
# create the classifier and train with train df
classifier = BayesClassifier()
classifier.fit(prep.train)

# predict for young age (0) and low income (1)
prediction = classifier.predict(0, 1)
print(prediction)

Camina


In [ ]:
# get all predictions for test df
y_pred = [classifier.predict(a, i) for a, i in zip(test["age-bin"], test["income-bin"])]

print("Predicción:", prediction)
print("Probabilidades:", y_pred)

Predicción: Camina
Probabilidades: ['Conduce', 'Conduce', 'Camina', 'Conduce', 'Conduce', 'Conduce', 'Camina', 'Camina', 'Conduce', 'Conduce', 'Conduce', 'Camina', 'Conduce', 'Conduce', 'Camina', 'Conduce', 'Conduce', 'Camina', 'Conduce', 'Conduce']


## 3. Evaluación del Modelo

In [ ]:
class Evaluator:

  def __init__(self, y_true, y_pred, classes=('Camina','Conduce')):
    self.y_pred = pd.Series(list(y_pred), name="Predicted")
    self.y_true = pd.Series(list(y_true), name="Real")
    self.classes = list(classes)

  def get_confusion_matrix(self):
    #          Predicted
    #          Pos   Neg
    #        +-----+-----+
    #    Pos |  TP |  FN |
    # Actual +-----+-----+
    #    Neg |  FP |  TN |
    #        +-----+-----+
    self.df_confusion = pd.crosstab(self.y_true, self.y_pred)
    self.df_confusion = self.df_confusion.reindex(index=self.classes, columns=self.classes, fill_value=0)
    return self.df_confusion

  def accuracy(self):
    # accuracy = success (diagonal sum) / total
    if self.df_confusion is None:
      self.confusion_matrix()

    return np.diag(self.df_confusion.to_numpy()).sum() / self.df_confusion.to_numpy().sum()

  def report(self):
    # print the confussion matrix and accuracy
    confusion_matrix = self.get_confusion_matrix()
    accuracy = self.accuracy()
    success = int(np.diag(confusion_matrix.to_numpy()).sum())
    total = int(confusion_matrix.to_numpy().sum())
    print("Confusion matrix (rows = real, columns = predicted):")
    print(confusion_matrix)
    print()
    print("Aciertos: {} de {}".format(success, total))
    print("Precisión general: {:.2%}".format(accuracy))

In [ ]:
evaluator = Evaluator(test["Clase"], y_pred)
evaluator.report()

Confusion matrix (rows = real, columns = predicted):
Predicted  Camina  Conduce
Real                      
Camina          3        0
Conduce         3       14

Aciertos: 17 de 20
Precisión general: 85.00%
